In [1]:
import gensim.downloader as api

In [2]:
wv= api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [3]:
import pandas as pd
df= pd.read_csv('Fake.csv')

In [4]:
df.shape

(23481, 4)

In [5]:
df.head(4)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"


In [6]:
df= df.drop(columns=['title','date'])

In [8]:
df.head(2)

,text,subject
0,Donald Trump just couldn t wish all Americans ...,News
1,House Intelligence Committee Chairman Devin Nu...,News


In [9]:
df['subject'].value_counts()

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

In [10]:
df['subject_label']= df['subject'].map({'News':0,'politics':1,'left-news':2,'Government News':3,'US_News':4,'Middle-east':5})

In [13]:
df.sample(3)

,text,subject,subject_label
21054,When Sharia Law becomes the official law of th...,left-news,2
20435,The media will ignore the fact that this man w...,left-news,2
2771,GOP Congressman Darrell Issa offered the most ...,News,0


In [15]:
import spacy
nlp= spacy.load('en_core_web_sm')

In [17]:
def preprocess_and_vectorize (text):
    doc  = nlp(text)
    filtered_text=[]
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_text.append(token.lemma_)
    return wv.get_mean_vector(filtered_text)

In [19]:
df['vector'] = df['text'].apply(lambda text: preprocess_and_vectorize(text))

In [20]:
df.sample(3)

,text,subject,subject_label,vector
20416,Sounds like one of the best reasons ever TO vo...,left-news,2,"[-0.004282375, 0.010560414, 0.0015159383, 0.02..."
18320,"More than 21,000 people from all regions of th...",left-news,2,"[-0.0076082963, 0.012287965, -0.0002298479, 0...."
10591,Did President Trump visit Camp David this Fath...,politics,1,"[-0.0012604075, 0.01033311, -0.013398819, 0.01..."


In [23]:
from sklearn.model_selection import train_test_split


#Do the 'train-test' splitting with test size of 20% with random state of 2022 and stratify sampling too
X_train, X_test, y_train, y_test = train_test_split(
    df.vector.values, 
    df.subject_label, 
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022,
    stratify=df.subject_label
)

In [25]:
import numpy as np

In [26]:
print("Shape of X_train before reshaping: ", X_train.shape)
print("Shape of X_test before reshaping: ", X_test.shape)


X_train_2d = np.stack(X_train)
X_test_2d =  np.stack(X_test)

print("Shape of X_train after reshaping: ", X_train_2d.shape)
print("Shape of X_test after reshaping: ", X_test_2d.shape)

Shape of X_train before reshaping:  (18784,)
Shape of X_test before reshaping:  (4697,)
Shape of X_train after reshaping:  (18784, 300)
Shape of X_test after reshaping:  (4697, 300)


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

#1. creating a GradientBoosting model object
clf = GradientBoostingClassifier()

#2. fit with all_train_embeddings and y_train
clf.fit(X_train_2d, y_train)


#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = clf.predict(X_test_2d)


#4. print the classfication report
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
cm


from matplotlib import pyplot as plt
import seaborn as sn
plt.figure(figsize = (10,7))
sn.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Prediction')
plt.ylabel('Truth')
